# 03 · Run Experiments

Launch any of the experiment scripts in `experiments/` (or the master
orchestrator) with a clean interface. Output streams live into the notebook
and each script writes its own `*_report.json` + PNGs under `results/`.

Pick a row from the catalog, set the args, run the cell.

In [ ]:
import nb_config as C
C.setup()
DEVICE = C.detect_device()
N = 200    # images per split for the experiments below

## Experiment catalog
The most useful, paper-mapped scripts. (`experiments/` has ~30 total — list them with the next cell.)

In [ ]:
import os
catalog = {
    "noise_robustness":       "Metric monotonicity under Gaussian noise / blur (Sec 3.5)",
    "ood_detection":          "OOD / anomaly detection AUC: M3 vs FID (Sec 3.6)",
    "hallucination_detection":"Pathology-masking sensitivity, M3 vs FID (Sec 3.11)",
    "interpretability":       "Radiomic + RadioDino feature decomposition (Sec 3.10)",
    "feature_orthogonality":  "Non-redundancy of the 3 scales (Sec 3.10)",
    "weight_ablation":        "Optimal layer weights ablation (Sec 3.8)",
    "permutation_test":       "Statistical significance via null distribution (Sec 3.7)",
    "n_scaling":              "Sample-size stability / coefficient of variation",
    "noise_quality_ladder":   "M3/FID/KID/CMMD vs increasing noise sigma",
}
for k, v in catalog.items():
    exists = os.path.exists(os.path.join(C.EXP_DIR, k + ".py"))
    print(f"  [{'x' if exists else ' '}] {k:<26} {v}")

print("\nAll scripts in experiments/:")
print("  " + ", ".join(sorted(f[:-3] for f in os.listdir(C.EXP_DIR)
                              if f.endswith('.py') and not f.startswith('_'))))

## Run a single experiment
Most scripts accept `--real_dir --gen_dir --output_dir --num_images --device`.
Noise/robustness scripts only need `--real_dir` (they degrade the reals themselves).

In [ ]:
EXPERIMENT = "noise_robustness"          # <- change me
OUT = f"results/notebook/{EXPERIMENT}"

args = [
    "--real_dir",   C.REAL_DIR,
    "--output_dir", OUT,
    "--num_images", str(N),
    "--device",     DEVICE,
]
# Scripts that also need generated images:
if EXPERIMENT in {"ood_detection", "hallucination_detection",
                  "interpretability", "feature_orthogonality"}:
    args[2:2] = ["--gen_dir", C.GEN_DIR]

C.run_script(f"experiments/{EXPERIMENT}.py", args)

## Show what it produced
Loads the JSON report and renders any PNGs the script saved.

In [ ]:
import os, glob
out_abs = os.path.join(C.PROJECT_ROOT, OUT)
reports = glob.glob(os.path.join(out_abs, "**", "*.json"), recursive=True)
for r in reports:
    print("==", os.path.relpath(r, C.PROJECT_ROOT))
    try:
        d = C.load_json(r)
        keys = list(d.keys()) if isinstance(d, dict) else f"list[{len(d)}]"
        print("   keys:", keys)
    except Exception as e:
        print("   (could not parse)", e)

pngs = glob.glob(os.path.join(out_abs, "**", "*.png"), recursive=True)
C.show_pngs(pngs, ncols=2, max_imgs=6)

## Master orchestrator (everything at once)
Runs the whole suite. This is **slow** — uncomment only when you mean it.

In [ ]:
# C.run_script("evaluation/run_experiments.py", [
#     "--real_dir",   C.REAL_DIR,
#     "--gen_dir",    C.GEN_DIR,
#     "--output_dir", "results/notebook_full",
#     "--experiments", "all",
#     "--num_images", str(N),
#     "--device",     DEVICE,
# ])